# 04 — Model Training

## Objective
Train and compare:
1. **Logistic Regression** — baseline/interpretable linear classifier.
2. **Random Forest** — final nonlinear tree-based classifier.

Because failure events are less common than normal events, `class_weight="balanced"` is used to make the model pay more attention to the minority class.

The final model is selected using validation metrics such as **F1, recall, ROC-AUC and PR-AUC**, not accuracy alone.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

X_train = pd.read_csv(PROCESSED_DIR / "X_train.csv")
X_test = pd.read_csv(PROCESSED_DIR / "X_test.csv")
y_train = pd.read_csv(PROCESSED_DIR / "y_train.csv").squeeze("columns")
y_test = pd.read_csv(PROCESSED_DIR / "y_test.csv").squeeze("columns")

preprocessor = joblib.load(MODEL_DIR / "preprocessor.pkl")

X_train_p = preprocessor.transform(X_train)
X_test_p = preprocessor.transform(X_test)

print("Training matrix:", X_train_p.shape)

Training matrix: (9600, 12)


In [2]:
# Define models
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=400,
        max_depth=12,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    )
}

results = []

for name, model in models.items():
    model.fit(X_train_p, y_train)
    probability = model.predict_proba(X_test_p)[:, 1]
    prediction = (probability >= 0.50).astype(int)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, prediction),
        "Precision": precision_score(y_test, prediction, zero_division=0),
        "Recall": recall_score(y_test, prediction, zero_division=0),
        "F1": f1_score(y_test, prediction, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, probability),
        "PR_AUC": average_precision_score(y_test, probability)
    })

results_df = pd.DataFrame(results).sort_values("F1", ascending=False)
display(results_df)

,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
1,Random Forest,0.937917,0.722826,0.849840,0.781204,0.976212,0.868796
0,Logistic Regression,0.913750,0.609959,0.939297,0.739623,0.978612,0.891515


In [3]:
# Train and save the final Random Forest model
final_model = models["Random Forest"]
final_model.fit(X_train_p, y_train)

joblib.dump(final_model, MODEL_DIR / "random_forest_model.pkl")

print("Final model saved to:", MODEL_DIR / "random_forest_model.pkl")

Final model saved to: D:\Python_Programming\College Project\AI-Based Predictive Maintenance & Equipment Failure Detection System\ML_Predictive_Maintenance Code\models\random_forest_model.pkl


## Why Random Forest?

Random Forest is suitable for this project because equipment failure can depend on **nonlinear relationships and interactions** between temperature, vibration, tool wear, operating hours, electrical variables, and other conditions.

This does not mean Random Forest is universally the best algorithm. It is the model selected for this academic project based on the evaluation results generated above.


## Model Selection

### Logistic Regression
Used as a baseline classifier. It provides a relatively simple
model for comparing the more complex classifier.

### Random Forest
Uses multiple decision trees to learn potentially nonlinear
relationships between machine operating features and the target.

### Class Imbalance
Class weighting is used to give additional importance to the
minority class during training.

### Model Comparison
Compare the models using the metrics produced by the notebook:

- Precision
- Recall
- F1-score
- ROC-AUC
- PR-AUC

### Final Model
The project saves the selected Random Forest model after training.
The final selection should be justified using actual evaluation
results, not an assumption that Random Forest is always better.
